# Этап 2: baseline RandomForest (автономный)

Setup + auto-fetch dataset + обучение RF + графики MAE.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"  # при необходимости замените
REPO_DIR = Path("IR_expert_system_3")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd IR_expert_system_3
!pip install -q -e ".[torch]"
!ir-pipeline --help
import subprocess
help_txt = subprocess.check_output(['ir-pipeline', '--help'], text=True)
if ' run ' not in help_txt:
    print('WARNING: команда `run` отсутствует. Ноутбук использует fallback без run-stage.')


In [ ]:
from pathlib import Path

DATASET_DIR = Path('data/processed/dataset_mini')
if not DATASET_DIR.exists():
    print('dataset_mini not found → fetching from HF...')
    !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed
else:
    print(f'{DATASET_DIR} already exists')


In [ ]:
from pathlib import Path
import zipfile, shutil

SEARCH_ROOTS = [Path('/content'), Path('/content/IR_expert_system_3'), Path('/content/drive/MyDrive')]
candidates = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob('*'):
        name = p.name.lower()
        if p.is_dir() and name == 'downloaded_jcamp':
            candidates.append(('dir', p))
        if p.is_file() and ('downloaded_jcamp' in name and name.endswith('.zip')):
            candidates.append(('zip', p))

print('Found candidates:')
for k, p in candidates[:30]:
    print(k, p)

target = Path('/content/IR_expert_system_3/downloaded_jcamp')
target.parent.mkdir(parents=True, exist_ok=True)
if not target.exists():
    for kind, p in candidates:
        if kind == 'dir':
            print('Copying directory to', target)
            shutil.copytree(p, target, dirs_exist_ok=True)
            break
        if kind == 'zip':
            print('Extracting zip to', target)
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(p) as zf:
                zf.extractall(target)
            break
print('downloaded_jcamp exists:', target.exists())


In [ ]:
from pathlib import Path
from IPython.display import Image, display

RUN_DIR = Path('runs/colab_pipeline_rf/rf_run')
# GPU (опционально):
# !pip install -q -e ".[cuml]"
# import os; os.environ['IR_RF_BACKEND'] = 'cuml'

!ir-pipeline train --paths configs/paths.huggingface.yaml --dataset-version dataset_mini --mode spectrum --config configs/train_mini.yaml --run-dir {RUN_DIR}
!ir-pipeline plot-train-metrics --run-dir {RUN_DIR}

for pat in ['metrics_per_band_mae.png', 'metrics_by_group_mae.png']:
    hits = sorted(Path('runs').rglob(pat))
    if hits:
        print(hits[-1])
        display(Image(filename=str(hits[-1]), width=900))
